## Homework 3: Symbolic Music Generation Using Markov Chains

**Before starting the homework:**

Please run `pip install miditok` to install the [MiDiTok](https://github.com/Natooz/MidiTok) package, which simplifies MIDI file processing by making note and beat extraction more straightforward.

You’re also welcome to experiment with other MIDI processing libraries such as [mido](https://github.com/mido/mido), [pretty_midi](https://github.com/craffel/pretty-midi) and [miditoolkit](https://github.com/YatingMusic/miditoolkit). However, with these libraries, you’ll need to handle MIDI quantization yourself, for example, converting note-on/note-off events into beat positions and durations.

In [11]:
# run this command to install MiDiTok
! pip install miditok


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [12]:
# import required packages
import random
from glob import glob
from collections import defaultdict

import numpy as np
from numpy.random import choice

from symusic import Score
from miditok import REMI, TokenizerConfig
from midiutil import MIDIFile

In [13]:
# You can change the random seed but try to keep your results deterministic!
# If I need to make changes to the autograder it'll require rerunning your code,
# so it should ideally generate the same results each time.
random.seed(42)

In [14]:
# from google.colab import drive
# drive.mount('/content/drive')

import zipfile
import os

if not os.path.exists("PDMX_subset"):
    with zipfile.ZipFile("PDMX_subset.zip", "r") as zip_ref:
        zip_ref.extractall(".")

### Load music dataset
We will use a subset of the [PDMX dataset](https://zenodo.org/records/14984509).

Please find the file `PDMX_subset.zip` in the homework spec.

All pieces are monophonic music (i.e. one melody line) in 4/4 time signature.

In [15]:
midi_files = glob('PDMX_subset/*.mid')
len(midi_files)

1000

### Train a tokenizer with the REMI method in MidiTok

In [16]:
config = TokenizerConfig(num_velocities=1, use_chords=False, use_programs=False)
tokenizer = REMI(config)
tokenizer.train(vocab_size=1000, files_paths=midi_files)

### Use the trained tokenizer to get tokens for each midi file
In REMI representation, each note will be represented with four tokens: `Position, Pitch, Velocity, Duration`, e.g. `('Position_28', 'Pitch_74', 'Velocity_127', 'Duration_0.4.8')`; a `Bar_None` token indicates the beginning of a new bar.

In [17]:
# e.g.:
midi = Score(midi_files[0])
tokens = tokenizer(midi)[0].tokens
tokens[:10]

['Bar_None',
 'Position_0',
 'Pitch_66',
 'Velocity_127',
 'Duration_1.0.8',
 'Position_8',
 'Pitch_66',
 'Velocity_127',
 'Duration_0.2.8',
 'Position_10']

1. Write a function to extract note pitch events from a midi file; and another extract all note pitch events from the dataset and output a dictionary that maps note pitch events to the number of times they occur in the files. (e.g. {60: 120, 61: 58, …}).

`note_extraction()`
- **Input**: a midi file

- **Output**: a list of note pitch events (e.g. [60, 62, 61, ...])

`note_frequency()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to the number of times they occur, e.g {60: 120, 61: 58, …}

In [18]:
def note_extraction(midi_file):
    # Q1a: Your code goes here
    midi = Score(midi_file)
    tokens = tokenizer(midi)[0].tokens

    notes = []
    for token in tokens:
        if token.startswith("Pitch_"):
            notes.append(int(token.split("_")[1]))

    return notes

In [19]:
def note_frequency(midi_files):
    # Q1b: Your code goes here
    note_counts = defaultdict(int)

    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for note in notes:
            note_counts[note] += 1

    return dict(note_counts)

2. Write a function to normalize the above dictionary to produce probability scores (e.g. {60: 0.13, 61: 0.065, …})

`note_unigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to probabilities, e.g. {60: 0.13, 61: 0.06, …}

In [20]:
def note_unigram_probability(midi_files):
    note_counts = note_frequency(midi_files)
    unigramProbabilities = {}

    # Q2: Your code goes here
    # ...

    total_notes = sum(note_counts.values())
    if total_notes == 0:
        return unigramProbabilities

    for note, count in note_counts.items():
        unigramProbabilities[note] = count / total_notes

    return unigramProbabilities

3. Generate a table of pairwise probabilities containing p(next_note | previous_note) values for the dataset; write a function that randomly generates the next note based on the previous note based on this distribution.

`note_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramTransitions`: key: previous_note, value: a list of next_note, e.g. {60:[62, 64, ..], 62:[60, 64, ..], ...} (i.e., this is a list of every other note that occured after note 60, every note that occured after note 62, etc.)

  - `bigramTransitionProbabilities`: key:previous_note, value: a list of probabilities for next_note in the same order of `bigramTransitions`, e.g. {60:[0.3, 0.4, ..], 62:[0.2, 0.1, ..], ...} (i.e., you are converting the values above to probabilities)

`sample_next_note()`
- **Input**: a note

- **Output**: next note sampled from pairwise probabilities

In [21]:
def note_bigram_probability(midi_files):
    bigramTransitions = defaultdict(list)
    bigramTransitionProbabilities = defaultdict(list)

    # Q3a: Your code goes here
    # ...

    bigram_counts = defaultdict(lambda: defaultdict(int))

    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for previous_note, next_note in zip(notes[:-1], notes[1:]):
            bigram_counts[previous_note][next_note] += 1

    for previous_note, next_counts in bigram_counts.items():
        total = sum(next_counts.values())
        for next_note, count in next_counts.items():
            bigramTransitions[previous_note].append(next_note)
            bigramTransitionProbabilities[previous_note].append(count / total)

    return bigramTransitions, bigramTransitionProbabilities

In [22]:
def sample_next_note(note):
    # Q3b: Your code goes here
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)

    if note in bigramTransitions and len(bigramTransitions[note]) > 0:
        return random.choices(
            list(bigramTransitions[note]),
            weights=list(bigramTransitionProbabilities[note]),
            k=1
        )[0]

    unigramProbabilities = note_unigram_probability(midi_files)
    return random.choices(
        list(unigramProbabilities.keys()),
        weights=list(unigramProbabilities.values()),
        k=1
    )[0]

4. Write a function to calculate the perplexity of your model on a midi file.

    The perplexity of a model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-1})))$

    where $p(w_1|w_0) = p(w_1)$, $p(w_i|w_{i-1}) (i>1)$ refers to the pairwise probability p(next_note | previous_note).

`note_bigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [23]:
def note_bigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)

    # Q4: Your code goes here
    # Can use regular numpy.log (i.e., natural logarithm)

    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return float("inf")

    epsilon = 1e-12
    log_probability_sum = 0.0

    for i, note in enumerate(notes):
        if i == 0:
            probability = unigramProbabilities.get(note, epsilon)
        else:
            previous_note = notes[i - 1]

            if previous_note in bigramTransitions and note in bigramTransitions[previous_note]:
                index = bigramTransitions[previous_note].index(note)
                probability = bigramTransitionProbabilities[previous_note][index]
            else:
                probability = epsilon

        log_probability_sum += np.log(max(probability, epsilon))

    return np.exp(-log_probability_sum / len(notes))

5. Implement a second-order Markov chain, i.e., one which estimates p(next_note | next_previous_note, previous_note); write a function to compute the perplexity of this new model on a midi file.

    The perplexity of this model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-2}, w_{i-1})))$

    where $p(w_1|w_{-1}, w_0) = p(w_1)$, $p(w_2|w_0, w_1) = p(w_2|w_1)$, $p(w_i|w_{i-2}, w_{i-1}) (i>2)$ refers to the probability p(next_note | next_previous_note, previous_note).


`note_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramTransitions`: key - (next_previous_note, previous_note), value - a list of next_note, e.g. {(60, 62):[64, 66, ..], (60, 64):[60, 64, ..], ...}

  - `trigramTransitionProbabilities`: key: (next_previous_note, previous_note), value: a list of probabilities for next_note in the same order of `trigramTransitions`, e.g. {(60, 62):[0.2, 0.2, ..], (60, 64):[0.4, 0.1, ..], ...}

`note_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [24]:
def note_trigram_probability(midi_files):
    trigramTransitions = defaultdict(list)
    trigramTransitionProbabilities = defaultdict(list)

    # Q5a: Your code goes here
    # ...

    trigram_counts = defaultdict(lambda: defaultdict(int))

    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for i in range(2, len(notes)):
            previous_two_notes = (notes[i - 2], notes[i - 1])
            next_note = notes[i]
            trigram_counts[previous_two_notes][next_note] += 1

    for previous_two_notes, next_counts in trigram_counts.items():
        total = sum(next_counts.values())
        for next_note, count in next_counts.items():
            trigramTransitions[previous_two_notes].append(next_note)
            trigramTransitionProbabilities[previous_two_notes].append(count / total)

    return trigramTransitions, trigramTransitionProbabilities

In [25]:
def note_trigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    # Q5b: Your code goes here

    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return float("inf")

    epsilon = 1e-12
    log_probability_sum = 0.0

    for i, note in enumerate(notes):
        if i == 0:
            probability = unigramProbabilities.get(note, epsilon)

        elif i == 1:
            previous_note = notes[i - 1]

            if previous_note in bigramTransitions and note in bigramTransitions[previous_note]:
                index = bigramTransitions[previous_note].index(note)
                probability = bigramTransitionProbabilities[previous_note][index]
            else:
                probability = epsilon

        else:
            previous_two_notes = (notes[i - 2], notes[i - 1])

            if previous_two_notes in trigramTransitions and note in trigramTransitions[previous_two_notes]:
                index = trigramTransitions[previous_two_notes].index(note)
                probability = trigramTransitionProbabilities[previous_two_notes][index]
            else:
                probability = epsilon

        log_probability_sum += np.log(max(probability, epsilon))

    return np.exp(-log_probability_sum / len(notes))

6. Our model currently doesn’t have any knowledge of beats. Write a function that extracts beat lengths and outputs a list of [(beat position; beat length)] values.

    Recall that each note will be encoded as `Position, Pitch, Velocity, Duration` using REMI. Please keep the `Position` value for beat position, and convert `Duration` to beat length using provided lookup table `duration2length` (see below).

    For example, for a note represented by four tokens `('Position_24', 'Pitch_72', 'Velocity_127', 'Duration_0.4.8')`, the extracted (beat position; beat length) value is `(24, 4)`.

    As a result, we will obtain a list like [(0,8),(8,16),(24,4),(28,4),(0,4)...], where the next beat position is the previous beat position + the beat length. As we divide each bar into 32 positions by default, when reaching the end of a bar (i.e. 28 + 4 = 32 in the case of (28, 4)), the beat position reset to 0.

In [26]:
duration2length = {
    '0.2.8': 2,  # sixteenth note, 0.25 beat in 4/4 time signature
    '0.4.8': 4,  # eighth note, 0.5 beat in 4/4 time signature
    '1.0.8': 8,  # quarter note, 1 beat in 4/4 time signature
    '2.0.8': 16, # half note, 2 beats in 4/4 time signature
    '4.0.4': 32, # whole note, 4 beats in 4/4 time signature
}

`beat_extraction()`
- **Input**: a midi file

- **Output**: a list of (beat position; beat length) values

In [27]:
def beat_extraction(midi_file):
    # Q6: Your code goes here
    midi = Score(midi_file)
    tokens = tokenizer(midi)[0].tokens

    beats = []
    current_position = None

    for token in tokens:
        if token.startswith("Position_"):
            current_position = int(token.split("_")[1])

        elif token.startswith("Duration_") and current_position is not None:
            duration_token = token.split("_", 1)[1]

            if duration_token in duration2length:
                beat_length = duration2length[duration_token]
                beats.append((current_position, beat_length))

    return beats

7. Implement a Markov chain that computes p(beat_length | previous_beat_length) based on the above function.

`beat_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatTransitions`: key: previous_beat_length, value: a list of beat_length, e.g. {4:[8, 2, ..], 8:[8, 4, ..], ...}

  - `bigramBeatTransitionProbabilities`: key - previous_beat_length, value - a list of probabilities for beat_length in the same order of `bigramBeatTransitions`, e.g. {4:[0.3, 0.2, ..], 8:[0.4, 0.4, ..], ...}

In [28]:
def beat_bigram_probability(midi_files):
    bigramBeatTransitions = defaultdict(list)
    bigramBeatTransitionProbabilities = defaultdict(list)

    # Q7: Your code goes here
    # ...

    bigram_counts = defaultdict(lambda: defaultdict(int))

    for midi_file in midi_files:
        beats = beat_extraction(midi_file)
        beat_lengths = [beat_length for beat_position, beat_length in beats]

        for previous_beat_length, next_beat_length in zip(beat_lengths[:-1], beat_lengths[1:]):
            bigram_counts[previous_beat_length][next_beat_length] += 1

    for previous_beat_length, next_counts in bigram_counts.items():
        total = sum(next_counts.values())
        for next_beat_length, count in next_counts.items():
            bigramBeatTransitions[previous_beat_length].append(next_beat_length)
            bigramBeatTransitionProbabilities[previous_beat_length].append(count / total)

    return bigramBeatTransitions, bigramBeatTransitionProbabilities

8. Implement a function to compute p(beat length | beat position), and compute the perplexity of your models from Q7 and Q8. For both models, we only consider the probabilities of predicting the sequence of **beat lengths**.

`beat_pos_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatPosTransitions`: key - beat_position, value - a list of beat_length

  - `bigramBeatPosTransitionProbabilities`: key - beat_position, value - a list of probabilities for beat_length in the same order of `bigramBeatPosTransitions`

`beat_bigram_perplexity()`
- **Input**: a midi file

- **Output**: two perplexity values correspond to the models in Q7 and Q8, respectively

In [29]:
def beat_pos_bigram_probability(midi_files):
    bigramBeatPosTransitions = defaultdict(list)
    bigramBeatPosTransitionProbabilities = defaultdict(list)

    # Q8a: Your code goes here
    # ...

    position_counts = defaultdict(lambda: defaultdict(int))

    for midi_file in midi_files:
        beats = beat_extraction(midi_file)

        for beat_position, beat_length in beats:
            position_counts[beat_position][beat_length] += 1

    for beat_position, length_counts in position_counts.items():
        total = sum(length_counts.values())
        for beat_length, count in length_counts.items():
            bigramBeatPosTransitions[beat_position].append(beat_length)
            bigramBeatPosTransitionProbabilities[beat_position].append(count / total)

    return bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities

In [30]:
def beat_bigram_perplexity(midi_file):
    bigramBeatTransitions, bigramBeatTransitionProbabilities = beat_bigram_probability(midi_files)
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    # Q8b: Your code goes here
    # Hint: one more probability function needs to be computed

    beat_unigram_counts = defaultdict(int)
    for train_file in midi_files:
        for beat_position, beat_length in beat_extraction(train_file):
            beat_unigram_counts[beat_length] += 1

    total_beats = sum(beat_unigram_counts.values())
    beatUnigramProbabilities = {}
    for beat_length, count in beat_unigram_counts.items():
        beatUnigramProbabilities[beat_length] = count / total_beats

    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return float("inf"), float("inf")

    epsilon = 1e-12
    log_probability_sum_Q7 = 0.0
    log_probability_sum_Q8 = 0.0

    for i, (beat_position, beat_length) in enumerate(beats):
        if i == 0:
            probability_Q7 = beatUnigramProbabilities.get(beat_length, epsilon)
        else:
            previous_beat_length = beats[i - 1][1]

            if previous_beat_length in bigramBeatTransitions and beat_length in bigramBeatTransitions[previous_beat_length]:
                index = bigramBeatTransitions[previous_beat_length].index(beat_length)
                probability_Q7 = bigramBeatTransitionProbabilities[previous_beat_length][index]
            else:
                probability_Q7 = epsilon

        if beat_position in bigramBeatPosTransitions and beat_length in bigramBeatPosTransitions[beat_position]:
            index = bigramBeatPosTransitions[beat_position].index(beat_length)
            probability_Q8 = bigramBeatPosTransitionProbabilities[beat_position][index]
        else:
            probability_Q8 = epsilon

        log_probability_sum_Q7 += np.log(max(probability_Q7, epsilon))
        log_probability_sum_Q8 += np.log(max(probability_Q8, epsilon))

    perplexity_Q7 = np.exp(-log_probability_sum_Q7 / len(beats))
    perplexity_Q8 = np.exp(-log_probability_sum_Q8 / len(beats))

    # # perplexity for Q7
    # perplexity_Q7 = None

    # # perplexity for Q8
    # perplexity_Q8 = None

    return perplexity_Q7, perplexity_Q8

9. Implement a Markov chain that computes p(beat_length | previous_beat_length, beat_position), and report its perplexity.

`beat_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramBeatTransitions`: key: (previous_beat_length, beat_position), value: a list of beat_length

  - `trigramBeatTransitionProbabilities`: key: (previous_beat_length, beat_position), value: a list of probabilities for beat_length in the same order of `trigramBeatTransitions`

`beat_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [31]:
def beat_trigram_probability(midi_files):
    trigramBeatTransitions = defaultdict(list)
    trigramBeatTransitionProbabilities = defaultdict(list)

    # Q9a: Your code goes here
    # ...

    trigram_counts = defaultdict(lambda: defaultdict(int))

    for midi_file in midi_files:
        beats = beat_extraction(midi_file)

        for i in range(1, len(beats)):
            previous_beat_length = beats[i - 1][1]
            beat_position = beats[i][0]
            beat_length = beats[i][1]

            trigram_counts[(previous_beat_length, beat_position)][beat_length] += 1

    for key, length_counts in trigram_counts.items():
        total = sum(length_counts.values())
        for beat_length, count in length_counts.items():
            trigramBeatTransitions[key].append(beat_length)
            trigramBeatTransitionProbabilities[key].append(count / total)

    return trigramBeatTransitions, trigramBeatTransitionProbabilities

In [32]:
def beat_trigram_perplexity(midi_file):
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    trigramBeatTransitions, trigramBeatTransitionProbabilities = beat_trigram_probability(midi_files)
    # Q9b: Your code goes here

    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return float("inf")

    epsilon = 1e-12
    log_probability_sum = 0.0

    for i, (beat_position, beat_length) in enumerate(beats):
        if i == 0:
            if beat_position in bigramBeatPosTransitions and beat_length in bigramBeatPosTransitions[beat_position]:
                index = bigramBeatPosTransitions[beat_position].index(beat_length)
                probability = bigramBeatPosTransitionProbabilities[beat_position][index]
            else:
                probability = epsilon
        else:
            previous_beat_length = beats[i - 1][1]
            key = (previous_beat_length, beat_position)

            if key in trigramBeatTransitions and beat_length in trigramBeatTransitions[key]:
                index = trigramBeatTransitions[key].index(beat_length)
                probability = trigramBeatTransitionProbabilities[key][index]
            else:
                probability = epsilon

        log_probability_sum += np.log(max(probability, epsilon))

    return np.exp(-log_probability_sum / len(beats))

10. Use the model from Q5 to generate N notes, and the model from Q8 to generate beat lengths for each note. Save the generated music as a midi file (see code from workbook1) as q10.mid. Remember to reset the beat position to 0 when reaching the end of a bar.

`music_generate`
- **Input**: target length, e.g. 500

- **Output**: a midi file q10.mid

Note: the duration of one beat in MIDIUtil is 1, while in MidiTok is 8. Divide beat length by 8 if you use methods in MIDIUtil to save midi files.

In [33]:
def music_generate(length):
    # sample notes
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    # Q10: Your code goes here ...
    sampled_notes = []

    if length > 0:
        first_note = random.choices(
            list(unigramProbabilities.keys()),
            weights=list(unigramProbabilities.values()),
            k=1
        )[0]
        sampled_notes.append(first_note)

    if length > 1:
        previous_note = sampled_notes[-1]

        if previous_note in bigramTransitions:
            second_note = random.choices(
                list(bigramTransitions[previous_note]),
                weights=list(bigramTransitionProbabilities[previous_note]),
                k=1
            )[0]
        else:
            second_note = random.choices(
                list(unigramProbabilities.keys()),
                weights=list(unigramProbabilities.values()),
                k=1
            )[0]

        sampled_notes.append(second_note)

    while len(sampled_notes) < length:
        key = (sampled_notes[-2], sampled_notes[-1])

        if key in trigramTransitions:
            next_note = random.choices(
                list(trigramTransitions[key]),
                weights=list(trigramTransitionProbabilities[key]),
                k=1
            )[0]
        elif sampled_notes[-1] in bigramTransitions:
            previous_note = sampled_notes[-1]
            next_note = random.choices(
                list(bigramTransitions[previous_note]),
                weights=list(bigramTransitionProbabilities[previous_note]),
                k=1
            )[0]
        else:
            next_note = random.choices(
                list(unigramProbabilities.keys()),
                weights=list(unigramProbabilities.values()),
                k=1
            )[0]

        sampled_notes.append(next_note)

    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)

    beat_unigram_counts = defaultdict(int)
    for train_file in midi_files:
        for beat_position, beat_length in beat_extraction(train_file):
            beat_unigram_counts[beat_length] += 1

    total_beats = sum(beat_unigram_counts.values())
    beatUnigramProbabilities = {}
    for beat_length, count in beat_unigram_counts.items():
        beatUnigramProbabilities[beat_length] = count / total_beats

    # sample beats
    sampled_beats = []

    beat_position = 0

    for i in range(length):
        if beat_position in bigramBeatPosTransitions:
            beat_length = random.choices(
                list(bigramBeatPosTransitions[beat_position]),
                weights=list(bigramBeatPosTransitionProbabilities[beat_position]),
                k=1
            )[0]
        else:
            beat_length = random.choices(
                list(beatUnigramProbabilities.keys()),
                weights=list(beatUnigramProbabilities.values()),
                k=1
            )[0]

        sampled_beats.append(beat_length)
        beat_position = (beat_position + beat_length) % 32

    # save the generated music as a midi file
    midi = MIDIFile(1)
    track = 0
    channel = 0
    time = 0
    tempo = 120
    volume = 100

    midi.addTempo(track, time, tempo)

    current_time = 0
    for note, beat_length in zip(sampled_notes, sampled_beats):
        duration = beat_length / 8
        midi.addNote(track, channel, int(note), current_time, duration, volume)
        current_time += duration

    with open("q10.mid", "wb") as f:
        midi.writeFile(f)
